# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fin1806/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

For Lane 2, the task type is scoring and ranking. The question I'm trying to answer is which pages should editors look at first, not just whether a page is declining or not. The output is a number assigned to each page that reflects how urgently it needs attention, so the team can sort by it and work from the top down.

I chose scoring over classification because a yes/no prediction doesn't actually solve the business problem. Flagging all declining pages gives you 16,000 rows with no way to prioritize. A ranked score lets the content team cap their review at whatever they can handle that week, like the top 50 or top 100, and still be confident they are working the highest-priority items first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target I'll use is is_declining_label, a binary column that marks a page as declining (1) or not (0). This is an observed outcome because trend_pct actually measures the percentage change in clicks over the trailing 90-day window, and the label comes from that real measurement, not from a rule I made up.

The label trap to watch out for: trend_direction and trend_pct encode the label directly, so they can never be used as features. Everything else in the dataset is fair game after checking for other leakage.

I'm using this label instead of building a composite score from scratch because a composite would reflect my own definition of "worth reviewing," not something the data actually observed. Pages that genuinely lost clicks is a cleaner, more honest starting point.

In [11]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Label distribution:")
print(df['is_declining_label'].value_counts())

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric I'll use is Precision at K, where K is 100. This asks: of the top 100 pages the model surfaces, how many actually needed attention? I picked K=100 because that's roughly what a content team could realistically review in one cycle.

I chose this over AUC because AUC measures how well the model ranks across the entire dataset, but the team will never review 30,000 pages. What matters is whether the top of the ranked list is actually useful. If the model puts the right pages in the top 100, it's doing its job.

A random ranker would hit about 54% precision at K=100, since that's the base rate of declining pages. The model should beat that clearly. I'm setting 0.75 as a first target, which is roughly 1.4 times better than random.

In [12]:
K = 100
base_rate = df['is_declining_label'].mean()
target_precision_at_k = 0.75

print(f"Base rate (random Precision@K):  {base_rate:.3f}  ({base_rate:.1%})")
print(f"Target Precision@{K}: {target_precision_at_k}")
print(f"Lift needed over random: {target_precision_at_k / base_rate:.2f}x")

Base rate (random Precision@K):  0.542  (54.2%)
Target Precision@100: 0.75
Lift needed over random: 1.38x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row in this dataset is one content item, meaning one page or URL belonging to a client. There are 30,000 items across 32 clients in total. The columns most relevant for Lane 2 are the ones that capture traffic volume, engagement, how stale the content is, and whether the page is declining.

In [13]:
key_cols = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'ctr',
    'avg_position', 'days_since_last_update',
    'is_declining_label'
]

print(f"Shape: {df.shape}\n")
df[key_cols].head()

Shape: (30000, 45)



,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like flagging every page where trend_direction is down gives you 16,262 results. That's more than any content team can realistically work through, so the rule doesn't actually help prioritize.

We could try to tighten it, like only flagging pages with high impressions and a declining trend, but then you have to decide what counts as high impressions. The right cutoff is different for a client with mostly low-traffic pages versus one with a large established site. There is no single threshold that works everywhere.

What actually matters is how the signals combine. A page that is declining, has high impressions, hasn't been updated in over a year, and has a poor click-through rate is a very different case from a page that is also declining but has low impressions and was refreshed last month. These combinations don't follow a clean rule, they shift by content type and client, and that's exactly the kind of problem where a model can learn something a hand-written if-statement can't capture.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.